# Simple Baseline Inference

Минимальный ноутбук: загрузить LLM с Hugging Face, дать ей заголовки из CSV и посмотреть сгенерированные описания.


In [1]:
# Если зависимости не установлены, раскомментируй и выполни один раз.
# !pip install -U transformers accelerate pandas tqdm
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


In [2]:
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
DATASET_PATH = Path("dataset_filtered_with_params.csv")
N_EXAMPLES = 8

if not DATASET_PATH.exists():
    DATASET_PATH = Path("..") / "dataset_filtered_with_params.csv"

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA: True
GPU: NVIDIA GeForce RTX 3060


In [3]:
df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=["title", "description"]).copy()
df["title"] = df["title"].astype(str).str.strip()
df["description"] = df["description"].astype(str).str.strip()

examples = df.sample(N_EXAMPLES, random_state=42).reset_index(drop=True)
display(examples[["title", "price", "category_id", "microcat_id"]])


,title,price,category_id,microcat_id
0,К.Славенски Дж.Д.Сэлинджер. Идя через рожь. 2012г,1100.0,52,627768250001
1,"Промтоварный фургон ГАЗ 2790, 2007",499000.0,14,5208
2,"Системный блок, Настольный пк Ryzen 5 600X",54990.0,36,405806250001
3,Велосипед бу,7000.0,4,598455000001
4,Экзотическая девочка,23000.0,31,2698
5,Xbox Series S,18999.0,13,531814500001
6,"Motoland XR 300 LITE, 2025, 200 300 км",150000.0,48,3674
7,"ВАЗ (LADA) Vesta 1.8 CVT, 2024, 39 000 км",1350000.0,43,6250020


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=False,
)

model.eval()
print("Model loaded:", MODEL_ID)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3-4B-Instruct-2507


In [5]:
SYSTEM_PROMPT = """
Ты русскоязычный помощник для объявлений. Пиши короткие живые описания как обычный продавец.
Главное: не искажай слова из заголовка. Если слово незнакомое, просто скопируй его без изменений.
""".strip()

USER_PROMPT_TEMPLATE = """
Напиши готовое описание объявления по заголовку.

Формат:
- 2 коротких предложения, примерно 80-220 символов;
- первое предложение начни с точного заголовка в исходном написании, затем добавь живое естественное продолжение;
- второе предложение коротко объясняет, кому или для чего это может подойти.

Как писать:
- сохраняй бренды, модели, числа, размеры и редкие слова ровно как в заголовке;
- не заменяй слова похожими: если в заголовке "Сапборд", нельзя писать "Саппорд", "Sapbord" или другое слово;
- можно добавлять только очевидное бытовое назначение по типу объявления;
- если заголовок похож на должность, вакансию или резюме, не описывай это как товар;
- если в заголовке есть "новый", "новая" или "новое", можно написать "новый" или "в отличном состоянии";
- не добавляй конкретные скрытые факты, если их нет в заголовке: комплект, гарантию, доставку, торг, адрес, дефекты, документы, материал, цвет, точный размер, возраст;
- не пиши цену, телефон, ссылки, "в наличии", "звоните", "пишите", "убедитесь", "вы нашли то, что искали";
- без markdown, списков, эмодзи, заголовков и пояснений.

Заголовок: {title}

Верни только текст описания.
""".strip()


def generate_description(row):
    title = str(row.get("title", "")).strip()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(title=title)},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.25,
            top_p=0.85,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


In [6]:
# for i, row in examples.iterrows():
#     generated = generate_description(row)

#     print("=" * 100)
#     print(f"ПРИМЕР {i + 1}")
#     print("Заголовок:", row["title"])
#     print("\nОтвет модели:")
#     print(generated)
#     print("\nОписание из датасета для сравнения:")
#     print(row["description"][:700])
#     print()


In [10]:
import os
import zipfile
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ==============================================================================
# 1. НАСТРОЙКА ПУТЕЙ
# ==============================================================================
ZIP_ARCHIVE_PATH = "qwen3-4b-avito-qlora.zip" 
EXTRACT_TO_DIR = "./qwen3-4b-avito-qlora"

if not os.path.exists(EXTRACT_TO_DIR):
    print(f"📦 Распаковка архива {ZIP_ARCHIVE_PATH}...")
    with zipfile.ZipFile(ZIP_ARCHIVE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_TO_DIR)
    print("✅ Распаковка завершена!")

ADAPTER_PATH = os.path.join(EXTRACT_TO_DIR, "qwen3-4b-avito-qlora") 
BASE_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# ==============================================================================
# 2. ЗАГРУЗКА ТОКЕНИЗАТОРА И МОДЕЛИ (ИСПРАВЛЕННЫЙ ЧИСТЫЙ ВАРИАНТ)
# ==============================================================================
print("🚀 Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

print("🧠 Загрузка базовой модели в оперативной памяти (bfloat16)...")
# ❌ МЫ УБРАЛИ device_map="auto"
# Модель загрузится сначала в обычную ОЗУ, не включая проблемные механизмы деления слоев.
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16
)

print("🔌 Подключение обученного LoRA-адаптера...")
# Теперь оригинальный метод отработает идеально и без конфликтов версий
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("🔥 Перенос объединенной модели на видеокарту CUDA...")
# Переносим всю готовую конструкцию (базу + адаптер) целиком на вашу RTX 3060
model = model.to("cuda")
model.eval() 

print("✨ Модель полностью готова к инференсу на GPU!")

🚀 Загрузка токенизатора...
🧠 Загрузка базовой модели в оперативной памяти (bfloat16)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

🔌 Подключение обученного LoRA-адаптера...
🔥 Перенос объединенной модели на видеокарту CUDA...
✨ Модель полностью готова к инференсу на GPU!


In [11]:
for i, row in examples.iterrows():
    generated = generate_description(row)

    print("=" * 100)
    print(f"ПРИМЕР {i + 1}")
    print("Заголовок:", row["title"])
    print("\nОтвет модели:")
    print(generated)
    print("\nОписание из датасета для сравнения:")
    print(row["description"][:700])
    print()


ПРИМЕР 1
Заголовок: К.Славенски Дж.Д.Сэлинджер. Идя через рожь. 2012г

Ответ модели:
Книга в хорошем состоянии. В книге есть несколько страниц с небольшими пятнами от времени.

Описание из датасета для сравнения:
Славенски К. Дж.Д.Сэлинджер. Идя через рожь. 

Перевод с английского (оригинальное название: "A life raised high"). Издательство "Колибри", 2012 год. Твердый глянцевый переплет. 496 страниц (Серия "Персона").Тираж 2500 экземпляров. Состояние хорошее

Книга - авторизованная биография Дж.Д.Селинджера.

ПРИМЕР 2
Заголовок: Промтоварный фургон ГАЗ 2790, 2007

Ответ модели:
Продаётся промтоварный фургон Газель 2790, 2007 года выпуска. В хорошем техническом состоянии, обслужен, работает все оборудование.

Описание из датасета для сравнения:
Продаю газель в очень достойном состоянии.кабина в хорошем состоянии, без коррозии .Чистый , не прокуренный салон . Двигатель простой и надёжный , змз 405 работает ровно . КПП в норме, рама целая без трещин , резина первый сезон , На данный момен

KeyboardInterrupt: 

In [8]:
# %pip install --upgrade transformers peft accelerate

In [ ]:
# --- Запуск инференса на всем датасете и сохранение результатов в output.csv
from pathlib import Path
from tqdm.auto import tqdm

# Убедимся, что используем именно valid_with_params.csv, если он есть
preferred = Path("valid_with_params.csv")
if preferred.exists():
    DATASET_PATH = preferred

# Если переменная df уже есть (загруженный датасет), используем её, иначе подгружаем
try:
    df
except NameError:
    df = pd.read_csv(DATASET_PATH)

# Гарантируем порядок, убираем строки без заголовка
if 'title' in df.columns:
    df = df.dropna(subset=['title']).reset_index(drop=True)
else:
    df = df.reset_index(drop=True)

output_path = Path("output.csv")
print(f"Начинаю генерацию для {len(df)} строк — сохраняю в: {output_path}")

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Inference all rows"):
    try:
        generated = generate_description(row)
    except Exception as e:
        generated = ""
    results.append(generated)

# Добавляем результаты к датафрейму и сохраняем
df["generated_description"] = results

df.to_csv(output_path, index=False)
print(f"Готово — сохранено {len(df)} строк в {output_path}")
